# Importing Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import polars as pl
import numpy as np
from google.colab import drive
import pandas as pd
from collections import defaultdict

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Defining File Paths

In [ ]:
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
foundation_path = f"{drive_path}/processed_data/final_foundation"
split_dir = f"{foundation_path}/splits"
variant_dir = f"{drive_path}/processed_data/variant_llm"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Load Datasets


In [ ]:

train_df = pl.read_parquet(f"{split_dir}/train.parquet")
val_df = pl.read_parquet(f"{split_dir}/val.parquet")
test_df = pl.read_parquet(f"{split_dir}/test.parquet")

Loading split data...


In [ ]:
# Loading Item Metadata and LLM Embeddings
items_df = pl.read_parquet(f"{split_dir}/items_metadata_final.parquet")
emb_df = pl.read_parquet(f"{variant_dir}/item_embeddings_768.parquet").sort("item_id")

In [ ]:
# Dynamic ID Counting; using +1 because IDs are 0-indexed
ACTUAL_NUM_USERS = int(train_df["user_id"].max() + 1)
ACTUAL_NUM_ITEMS = int(max(train_df["item_id"].max(), items_df["item_id"].max()) + 1)

In [ ]:
print(f"Users found in data: {ACTUAL_NUM_USERS:,}")
print(f"Items found in data: {ACTUAL_NUM_ITEMS:,}")

Users found in data: 411,375
Items found in data: 21,925


In [ ]:
# Align LLM Embedding Tensor; If the metadata has more items than the embedding file, we pad with zeros
raw_embeddings = np.stack(emb_df["embedding_llm"].to_list())
if raw_embeddings.shape[0] < ACTUAL_NUM_ITEMS:
    padding_rows = ACTUAL_NUM_ITEMS - raw_embeddings.shape[0]
    padding = np.zeros((padding_rows, 768))
    raw_embeddings = np.vstack([raw_embeddings, padding])
    print(f"Padded embedding tensor with {padding_rows} rows.")

item_embeddings_tensor = torch.tensor(raw_embeddings, dtype=torch.float32)
print(f"Final Embedding Tensor Shape: {item_embeddings_tensor.shape}")

Final Embedding Tensor Shape: torch.Size([21925, 768])


In [ ]:
class BoardGameDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_id"].to_numpy(), dtype=torch.long)
        self.items = torch.tensor(df["item_id"].to_numpy(), dtype=torch.long)
        self.ratings = torch.tensor(df["Rating"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

In [ ]:
BATCH_SIZE = 8192
train_loader = DataLoader(BoardGameDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(BoardGameDataset(val_df), batch_size=BATCH_SIZE)
test_loader = DataLoader(BoardGameDataset(test_df), batch_size=BATCH_SIZE)

Hybrid LLM Augmented Architecture

In [ ]:
class HybridLLMModel(nn.Module):
    def __init__(self, num_users, num_items, llm_embs, latent_dim=64):
        super().__init__()
        # Collaborative Towers
        self.user_emb = nn.Embedding(num_users, latent_dim)
        self.item_emb = nn.Embedding(num_items, latent_dim)

        # Frozen Semantic Tower
        self.register_buffer('llm_feat', llm_embs)

        # Hybrid MLP Head
        input_dim = latent_dim + latent_dim + 768
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, u_id, i_id):
        u_vec = self.user_emb(u_id)
        i_vec = self.item_emb(i_id)
        l_vec = self.llm_feat[i_id]

        combined = torch.cat([u_vec, i_vec, l_vec], dim=1)
        return self.mlp(combined).squeeze()

In [ ]:
model = HybridLLMModel(ACTUAL_NUM_USERS, ACTUAL_NUM_ITEMS, item_embeddings_tensor).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def run_evaluation(model, loader):
    model.eval()
    mse = 0
    with torch.no_grad():
        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            preds = model(u, i)
            mse += nn.functional.mse_loss(preds, r, reduction='sum').item()
    return np.sqrt(mse / len(loader.dataset))


# Training

In [ ]:
epochs = 5
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0
    for u, i, r in train_loader:
        u, i, r = u.to(device), i.to(device), r.to(device)

        optimizer.zero_grad()
        preds = model(u, i)
        loss = criterion(preds, r)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * u.size(0)

    train_rmse = np.sqrt(epoch_loss / len(train_loader.dataset))
    val_rmse = run_evaluation(model, val_loader)
    print(f"Epoch {epoch} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

# Final Save
torch.save(model.state_dict(), f"{variant_dir}/hybrid_llm_weights.pth")

Starting Training...
Epoch 1 | Train RMSE: 1.4087 | Val RMSE: 1.2330


KeyboardInterrupt: 

# Checking User Recommendations

In [ ]:
def recommend_for_user(user_name, k=10):
    # Map username to ID
    user_mapping = pl.read_csv(f"{drive_path}/processed_data/csv_backups/user_mapping.csv")
    user_row = user_mapping.filter(pl.col("Username") == user_name)

    if user_row.is_empty():
        return "User not found."

    u_idx = user_row["user_id"][0]

    model.eval()
    # Score all items for this user
    all_i_ids = torch.arange(ACTUAL_NUM_ITEMS).to(device)
    u_ids = torch.full((ACTUAL_NUM_ITEMS,), u_idx, dtype=torch.long).to(device)

    with torch.no_grad():
        scores = model(u_ids, all_i_ids).cpu().numpy()

    # Get Top K
    top_indices = np.argsort(scores)[-k:][::-1]

    # Map back to Game Names
    return items_df.filter(pl.col("item_id").is_in(top_indices)).select(["Name", "AvgRating"])

In [ ]:
print(recommend_for_user("007poptart"))

In [ ]:
print(recommend_for_user("0void0"))

# Retraining with increased batch size, and more epochs

In [ ]:
import time
OPT_BATCH_SIZE = 16384
train_loader = DataLoader(BoardGameDataset(train_df), batch_size=OPT_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(BoardGameDataset(val_df), batch_size=OPT_BATCH_SIZE, num_workers=2, pin_memory=True)

In [ ]:
# Setup Scheduler and Early Stopping Variables
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

best_val_rmse = float('inf')
early_stop_patience = 3
epochs_no_improve = 0
total_epochs = 15

In [ ]:
print(f"Starting Optimized Training (Batch Size: {OPT_BATCH_SIZE})...")
print(f"Tracking 'best_hybrid_llm_optimized.pth'\n")
for epoch in range(1, total_epochs + 1):
    start_time = time.time()

    # Train
    model.train()
    epoch_loss = 0
    for u, i, r in train_loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        optimizer.zero_grad()
        preds = model(u, i)
        loss = criterion(preds, r)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * u.size(0)

    train_rmse = np.sqrt(epoch_loss / len(train_loader.dataset))

    # Evaluate
    val_rmse = run_evaluation(model, val_loader)
    epoch_duration = (time.time() - start_time) / 60

    # Update Scheduler
    scheduler.step(val_rmse)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:02d} | Train: {train_rmse:.4f} | Val: {val_rmse:.4f} | LR: {current_lr:.6f} | Time: {epoch_duration:.2f}m")

    # Check for Improvement
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        epochs_no_improve = 0
        # Save as the new "Optimized" comparison model
        torch.save(model.state_dict(), f"{variant_dir}/best_hybrid_llm_optimized.pth")
        print(f"  --> New Best Model Saved!")
    else:
        epochs_no_improve += 1
        print(f"  --> No improvement for {epochs_no_improve} epoch(s).")

    # Early Stopping Logic
    if epochs_no_improve >= early_stop_patience:
        print(f"\n[Early Stopping] No improvement in {early_stop_patience} epochs. Training halted.")
        break

print(f"\n Training Complete. Best Val RMSE: {best_val_rmse:.4f}")

In [ ]:
model = HybridLLMModel(ACTUAL_NUM_USERS, ACTUAL_NUM_ITEMS, item_embeddings_tensor).to(device)

weights_path = f"{variant_dir}/best_hybrid_llm_optimized.pth"
model.load_state_dict(torch.load(weights_path))
model.eval()  # Set to evaluation mode for consistent inference
print(f"Model weights loaded from {weights_path}")

Model weights loaded from /content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/processed_data/variant_llm/best_hybrid_llm_optimized.pth


# Evaluation


In [ ]:
def calculate_hybrid_metrics_full(model, loader, k=10, threshold=7.0):
    """
    Calculates RMSE, Precision@K, Recall@K, and NDCG@K for the Hybrid PyTorch model.
    """
    model.eval()
    user_results = defaultdict(list)
    total_mse = 0

    print("Gathering predictions and calculating RMSE...")
    with torch.no_grad():
        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            preds = model(u, i)

            # 1. Accumulate Global MSE
            # We use reduction='sum' to get the total error for this batch
            total_mse += nn.functional.mse_loss(preds, r, reduction='sum').item()

            # 2. Store for Ranking Metrics
            u_np = u.cpu().numpy()
            r_np = r.cpu().numpy()
            p_np = preds.cpu().numpy()

            for idx in range(len(u_np)):
                user_results[int(u_np[idx])].append((float(p_np[idx]), float(r_np[idx])))

    # Final RMSE Calculation
    final_rmse = np.sqrt(total_mse / len(loader.dataset))

    precisions = []
    recalls = []
    ndcgs = []

    print("Computing ranking metrics per user...")
    for uid, ratings in user_results.items():
        # Sort by Predicted Rating (Model's Ranking)
        ratings.sort(key=lambda x: x[0], reverse=True)

        # Binary relevance for Precision/Recall
        n_rel = sum(1 for p, r in ratings if r >= threshold)
        n_rel_and_rec_k = sum(1 for p, r in ratings[:k] if r >= threshold)

        # Precision@K: hits in top K / K
        precisions.append(n_rel_and_rec_k / k)

        # Recall@K: hits in top K / total relevant in test set
        recalls.append(n_rel_and_rec_k / n_rel if n_rel > 0 else 0)

        # NDCG@K Calculation
        # DCG
        dcg = sum([r / np.log2(idx + 2) for idx, (p, r) in enumerate(ratings[:k])])

        # IDCG (Perfect sort based on actual ratings)
        ratings.sort(key=lambda x: x[1], reverse=True)
        idcg = sum([r / np.log2(idx + 2) for idx, (p, r) in enumerate(ratings[:k])])

        if idcg > 0:
            ndcgs.append(dcg / idcg)

    return {
        "RMSE": final_rmse,
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "NDCG@10": np.mean(ndcgs)
    }



In [ ]:
test_metrics = calculate_hybrid_metrics_full(model, test_loader)

print(f"\n--- FINAL HYBRID MODEL EVALUATION (Test Set) ---")
print(f"RMSE:         {test_metrics['RMSE']:.4f}")
print(f"Precision@10: {test_metrics['Precision@10']:.4f}")
print(f"Recall@10:    {test_metrics['Recall@10']:.4f}")
print(f"NDCG@10:      {test_metrics['NDCG@10']:.4f}")

Gathering predictions and calculating RMSE...
Computing ranking metrics per user...

--- FINAL HYBRID MODEL EVALUATION (Test Set) ---
RMSE:         1.2121
Precision@10: 0.2678
Recall@10:    0.9064
NDCG@10:      0.9851
